In [17]:
# 13_Lexicon_Harvest.ipynb
#
# PURPOSE:
#   Build a data-driven lexicon per CAPS subtopic by harvesting the 587 real
#   questions segmented across 2014-2022. Seeds come from CAPS subtopic names.
#
# INPUTS:
#   data/processed/taxonomy/taxonomy_math_v2.csv
#   data/processed/taxonomy/subtopics_math_v2.csv
#   data/interim/segmented/YYYY/exam_questions_YYYY.csv    (2014-2020)
#   data/processed/mapped/YYYY/question_topic_map_YYYY.csv (2021, 2022)
#
# OUTPUTS:
#   data/processed/taxonomy/lexicon_raw_corpus.csv
#   data/processed/taxonomy/lexicon_by_subtopic.csv
#   data/processed/taxonomy/lexicon_diagnostics.md
#
# DOES NOT:
#   - Run mapping
#   - Assign topics to questions
#   - Modify any existing CSV
#
# NEXT NOTEBOOK:
#   14_Memo_Signatures.ipynb

In [18]:
import re
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"
SEG_DIR = ROOT / "data" / "interim" / "segmented"
MAP_DIR = ROOT / "data" / "processed" / "mapped"

PATH_A_YEARS = [2020, 2019, 2018, 2017, 2016, 2015, 2014]
MAPPED_YEARS = [2021, 2022]

# English stopwords for mathematics corpus (curated, not full NLTK list)
STOPWORDS = set("""
a an the is are was were be been being am
to for of in on at by with from and or if it its
this that these those as so there their they them
we you he she i me my your our
what which who whom whose when where why how
do does did done doing
have has had having
will would shall should can could may might must
not no nor only also just very much many more most
then than such but any some all each every both
about into over under between among within without
via per eg ie etc
""".split())

# Common math OCR noise tokens (from the scanned papers)
MATH_NOISE = set("""
page dbe november nsc mathematics copyright reserved please turn over
nss marking guidelines nasienriglyne graad grade question vraag
total totaal marks punte answer antwoord marks punte
""".split())

In [19]:
tax = pd.read_csv(TAX_DIR / "taxonomy_math_v2.csv")
sub = pd.read_csv(TAX_DIR / "subtopics_math_v2.csv")
print(f"Topics:   {len(tax)}")
print(f"Subtopics: {len(sub)}")
print()

# Merge subtopic with topic info
sub = sub.merge(tax[["topic_id", "caps_name", "paper"]], on="topic_id", how="left")
print(sub.groupby("topic_id").size().to_string())

Topics:   10
Subtopics: 55

topic_id
AGEO    5
ALG     6
CALC    6
EUCL    6
FIN     5
FUNC    6
PROB    4
SEQ     5
STAT    6
TRIG    6


In [20]:
def load_path_a(year):
    p = SEG_DIR / str(year) / f"exam_questions_{year}.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p)
    df["source_set"] = "path_a"
    return df

def load_mapped(year):
    p = MAP_DIR / str(year) / f"question_topic_map_{year}.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p)
    df["source_set"] = "mapped_v1"
    return df

frames = []
for y in PATH_A_YEARS:
    df = load_path_a(y)
    if df is not None:
        frames.append(df)
        print(f"{y} (Path A): {len(df)} rows")

for y in MAPPED_YEARS:
    df = load_mapped(y)
    if df is not None:
        frames.append(df)
        print(f"{y} (mapped): {len(df)} rows")

corpus = pd.concat(frames, ignore_index=True, sort=False)
print(f"\nTotal corpus rows: {len(corpus)}")
print(f"Columns: {sorted(corpus.columns)}")

# Keep only rows with usable question text
corpus = corpus[corpus["question_text"].notna()].copy()
corpus = corpus[corpus["question_text"].astype(str).str.len() > 20].copy()
print(f"Rows with usable text: {len(corpus)}")

2020 (Path A): 107 rows
2019 (Path A): 99 rows
2018 (Path A): 124 rows
2017 (Path A): 100 rows
2016 (Path A): 82 rows
2015 (Path A): 78 rows
2014 (Path A): 82 rows
2021 (mapped): 119 rows
2022 (mapped): 116 rows

Total corpus rows: 907
Columns: ['data_quality', 'diagram_or_table_risk', 'diagram_present', 'document_id', 'extraction_method', 'load_bearing_diagram', 'mapped_at', 'mapping_confidence', 'mapping_method', 'marks', 'paper', 'question_number', 'question_text', 'rule_id', 'segmentation_status', 'source_set', 'source_type', 'subquestion', 'taxonomy_version', 'topic', 'vision_description', 'vision_verification_status', 'year']
Rows with usable text: 789


In [21]:
def tokenize(text: str):
    """Lowercase, keep alpha tokens length >= 3, drop stopwords + math noise."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   # keep letters only
    tokens = text.split()
    return [
        t for t in tokens
        if len(t) >= 3
        and t not in STOPWORDS
        and t not in MATH_NOISE
    ]

# Build token counts per row
corpus["tokens"] = corpus["question_text"].apply(tokenize)
corpus["n_tokens"] = corpus["tokens"].apply(len)

# Global token frequency across all rows
global_counter = Counter()
for tokens in corpus["tokens"]:
    global_counter.update(set(tokens))  # document frequency, not term frequency

raw = pd.DataFrame(
    global_counter.most_common(),
    columns=["token", "document_frequency"],
)
raw["corpus_size"] = len(corpus)
raw["df_ratio"] = (raw["document_frequency"] / len(corpus)).round(4)

out_raw = TAX_DIR / "lexicon_raw_corpus.csv"
raw.to_csv(out_raw, index=False)
print(f"Unique tokens: {len(raw)}")
print(f"Saved: {out_raw}")
print()
print("Top 60 tokens by document frequency:")
print(raw.head(60).to_string(index=False))

Unique tokens: 1281
Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\taxonomy\lexicon_raw_corpus.csv

Top 60 tokens by document frequency:
        token  document_frequency  corpus_size  df_ratio
    determine                 176          789    0.2231
    calculate                 166          789    0.2104
        value                  88          789    0.1115
         down                  87          789    0.1103
        write                  87          789    0.1103
        given                  77          789    0.0976
     equation                  66          789    0.0837
  coordinates                  59          789    0.0748
        first                  51          789    0.0646
       values                  50          789    0.0634
          cos                  50          789    0.0634
       circle                  50          789    0.0634
       number                  48          789    0.0608
      diagram               

In [22]:
def seed_tokens_from_name(name: str):
    """Turn a CAPS subtopic name into seed tokens."""
    text = name.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [
        t for t in tokens
        if len(t) >= 4
        and t not in STOPWORDS
        and t not in MATH_NOISE
    ]

sub["seed_tokens"] = sub["subtopic"].apply(seed_tokens_from_name)

# Diagnose: which subtopics have too-thin seeds?
sub["n_seeds"] = sub["seed_tokens"].apply(len)
thin = sub[sub["n_seeds"] < 2]
if len(thin) > 0:
    print("Subtopics with <2 seed tokens (need manual expansion):")
    print(thin[["topic_id", "subtopic", "seed_tokens"]].to_string(index=False))
else:
    print("All subtopics have >=2 seed tokens.")

Subtopics with <2 seed tokens (need manual expansion):
topic_id               subtopic    seed_tokens
    CALC Optimisation (max/min) [optimisation]


In [23]:
# For each subtopic, find tokens that appear most often in rows that contain
# at least one seed token. This is the "data-driven lexicon" per subtopic.
rows = []
for _, s in sub.iterrows():
    seeds = s["seed_tokens"]
    if not seeds:
        continue

    # Find matching rows: any row whose tokens contain any seed
    mask = corpus["tokens"].apply(lambda toks: any(seed in toks for seed in seeds))
    matched = corpus[mask]

    if len(matched) == 0:
        rows.append({
            "topic_id": s["topic_id"],
            "subtopic": s["subtopic"],
            "seed_tokens": "; ".join(seeds),
            "n_matched_questions": 0,
            "top_cooccurring_tokens": "",
        })
        continue

    co = Counter()
    for toks in matched["tokens"]:
        co.update(set(toks) - set(seeds))

    top = [f"{t}:{c}" for t, c in co.most_common(15)]

    rows.append({
        "topic_id": s["topic_id"],
        "subtopic": s["subtopic"],
        "seed_tokens": "; ".join(seeds),
        "n_matched_questions": len(matched),
        "top_cooccurring_tokens": "; ".join(top),
    })

lex_by_sub = pd.DataFrame(rows)
out = TAX_DIR / "lexicon_by_subtopic.csv"
lex_by_sub.to_csv(out, index=False)
print(f"Saved: {out}")
print()
print(lex_by_sub.to_string(index=False))

Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\taxonomy\lexicon_by_subtopic.csv

topic_id                                           subtopic                                        seed_tokens  n_matched_questions                                                                                                                                                    top_cooccurring_tokens
     ALG                                Exponents and surds                                   exponents; surds                    0                                                                                                                                                                          
     ALG                         Equations and inequalities                            equations; inequalities                    3                                                                                              down:2; asymptotes:2; write:2; lad:1; following:1; si

In [24]:
diag_path = TAX_DIR / "lexicon_diagnostics.md"

lines = ["# Lexicon Harvest Diagnostics", ""]
lines.append(f"Corpus size: {len(corpus)} questions across {corpus['year'].nunique()} years")
lines.append(f"Total tokens (document frequency): {len(raw)}")
lines.append("")

lines.append("## Subtopic coverage")
lines.append("")
lines.append("| topic_id | subtopic | n_matched | status |")
lines.append("|----------|----------|-----------|--------|")
for _, r in lex_by_sub.iterrows():
    n = r["n_matched_questions"]
    status = "OK" if n >= 5 else ("THIN" if n >= 1 else "MISSING")
    lines.append(f"| {r['topic_id']} | {r['subtopic']} | {n} | {status} |")
lines.append("")

lines.append("## Top 40 global tokens")
lines.append("")
lines.append("| token | df | df_ratio |")
lines.append("|-------|----|---------| ")
for _, r in raw.head(40).iterrows():
    lines.append(f"| {r['token']} | {r['document_frequency']} | {r['df_ratio']} |")

diag_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Saved: {diag_path}")
print()

# Quick summary
thin = lex_by_sub[lex_by_sub["n_matched_questions"] < 5]
missing = lex_by_sub[lex_by_sub["n_matched_questions"] == 0]
print(f"Subtopics with >=5 matches: {len(lex_by_sub) - len(thin)}/{len(lex_by_sub)}")
print(f"Subtopics thin (<5):        {len(thin)}")
print(f"Subtopics with 0 matches:   {len(missing)}")

Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\taxonomy\lexicon_diagnostics.md

Subtopics with >=5 matches: 37/55
Subtopics thin (<5):        18
Subtopics with 0 matches:   6


In [25]:
for f in ["lexicon_raw_corpus.csv", "lexicon_by_subtopic.csv", "lexicon_diagnostics.md"]:
    p = TAX_DIR / f
    print(f"{'OK ' if p.exists() else 'MISSING'} {f}")

OK  lexicon_raw_corpus.csv
OK  lexicon_by_subtopic.csv
OK  lexicon_diagnostics.md


In [26]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX = ROOT / "data" / "processed" / "taxonomy"

# Manual alias expansion (CAPS-aligned, exam language)
EXTRA = [
    # SEQ — sum to infinity
    {"topic_id": "SEQ", "subtopic": "Sum to infinity", "token": "sum to infinity"},
    {"topic_id": "SEQ", "subtopic": "Sum to infinity", "token": "converges"},
    {"topic_id": "SEQ", "subtopic": "Sum to infinity", "token": "infinite geometric"},
    {"topic_id": "SEQ", "subtopic": "Sum to infinity", "token": "s infinity"},
    {"topic_id": "SEQ", "subtopic": "Sum to infinity", "token": "|r|"},
    # FUNC — hyperbola
    {"topic_id": "FUNC", "subtopic": "Hyperbola", "token": "hyperbola"},
    {"topic_id": "FUNC", "subtopic": "Hyperbola", "token": "asymptote"},
    {"topic_id": "FUNC", "subtopic": "Hyperbola", "token": "xy ="},
    {"topic_id": "FUNC", "subtopic": "Hyperbola", "token": "y = k/x"},
    {"topic_id": "FUNC", "subtopic": "Hyperbola", "token": "vertical asymptote"},
    {"topic_id": "FUNC", "subtopic": "Hyperbola", "token": "horizontal asymptote"},
    # CALC — optimisation
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "optimisation"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "optimization"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "maximum"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "minimum"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "maximum profit"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "minimum cost"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "largest"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "smallest"},
    {"topic_id": "CALC", "subtopic": "Optimisation", "token": "rate of change"},
    # Common zero-match candidates (edit if your subtopic names differ)
    {"topic_id": "EUCL", "subtopic": "Similarity", "token": "similar triangles"},
    {"topic_id": "EUCL", "subtopic": "Similarity", "token": "proportional"},
    {"topic_id": "EUCL", "subtopic": "Circle geometry", "token": "angle in a semicircle"},
    {"topic_id": "EUCL", "subtopic": "Circle geometry", "token": "cyclic quadrilateral"},
    {"topic_id": "TRIG", "subtopic": "3D trigonometry", "token": "angle of elevation"},
    {"topic_id": "TRIG", "subtopic": "3D trigonometry", "token": "angle of depression"},
    {"topic_id": "AGEO", "subtopic": "Circle equation", "token": "equation of the circle"},
    {"topic_id": "AGEO", "subtopic": "Circle equation", "token": "centre and radius"},
    {"topic_id": "STAT", "subtopic": "Regression", "token": "least squares"},
    {"topic_id": "STAT", "subtopic": "Regression", "token": "correlation coefficient"},
    {"topic_id": "PROB", "subtopic": "Counting", "token": "fundamental counting"},
    {"topic_id": "PROB", "subtopic": "Counting", "token": "different codes"},
    {"topic_id": "FIN", "subtopic": "Annuities", "token": "present value"},
    {"topic_id": "FIN", "subtopic": "Annuities", "token": "future value"},
    {"topic_id": "ALG", "subtopic": "Inequalities", "token": "inequality"},
    {"topic_id": "ALG", "subtopic": "Inequalities", "token": "greater than"},
]

extra_df = pd.DataFrame(EXTRA)
out = TAX / "lexicon_manual_aliases_v2.csv"
extra_df.to_csv(out, index=False)
print("Alias rows:", len(extra_df))
print("Saved:", out)
print("\nAlign topic_id/subtopic strings to your subtopics_math_v2.csv names before rules.")

Alias rows: 36
Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\taxonomy\lexicon_manual_aliases_v2.csv

Align topic_id/subtopic strings to your subtopics_math_v2.csv names before rules.


In [27]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

sub = pd.read_csv(TAX_DIR / "subtopics_math_v2.csv")
print(f"Before: {len(sub)} subtopics")
print()

# Rename map — old CAPS name → exam-aligned name (same meaning, better seeds)
# Only applies where the CAPS spelling produces thin or zero matches.
RENAME = {
    "Sum to infinity":      "Infinite geometric series",
    "Hyperbola":            "Hyperbola (asymptotes)",
    "Optimisation":         "Optimisation (max/min)",
    "Box and whisker diagrams":         "Box and whisker plots",
    "Ogives (cumulative frequency)":    "Cumulative frequency (ogive)",
    "Parallel and perpendicular lines": "Gradient and parallel/perpendicular lines",
    "Line segment (gradient, midpoint, length)": "Coordinate geometry (gradient, midpoint, distance)",
    "2D and 3D problems (incl. heights and distances)": "2D and 3D trigonometric problems",
    "Circle geometry (centre, chord, tangent theorems)": "Circle geometry theorems",
    "Pythagorean theorem (proof and application)":       "Pythagoras (proof and application)",
}

# Apply the rename — only where key exists
before_names = sub["subtopic"].tolist()
sub["subtopic"] = sub["subtopic"].apply(lambda s: RENAME.get(s, s))

# Report what changed
changes = [(a, b) for a, b in zip(before_names, sub["subtopic"].tolist()) if a != b]
if changes:
    print("Renamed:")
    for a, b in changes:
        print(f"  {a}  →  {b}")
else:
    print("No renames applied.")

out = TAX_DIR / "subtopics_math_v2.csv"
sub.to_csv(out, index=False)
print()
print(f"Saved: {out}  ({len(sub)} rows)")

Before: 55 subtopics

No renames applied.

Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\taxonomy\subtopics_math_v2.csv  (55 rows)


In [28]:
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

# Signature patterns observed in NSC memos 2014-2022.
# Each signature: regex pattern + topic_id + confidence + notes.
# Patterns are case-insensitive and matched against memo text.
signatures = [
    # --- CALCULUS ---
    ("CALC", "first_principles",  r"f\s*'?\s*\(\s*x\s*\)\s*=\s*\\?lim",
        "high", "First principles definition"),
    ("CALC", "first_principles",  r"lim\s*_?\s*\{?\s*h\s*(→|->|\\to)",
        "high", "Limit notation"),
    ("CALC", "differentiate",     r"\\frac\s*\{\s*d\s*y\s*\}\s*\{\s*d\s*x\s*\}",
        "high", "dy/dx notation"),
    ("CALC", "differentiate",     r"D\s*_?\s*x\s*[\[\(]",
        "high", "D_x operator"),
    ("CALC", "cubic_graphs",      r"f\s*''?\s*\(\s*x\s*\)\s*=\s*0",
        "medium", "Turning point via f'(x)=0"),
    ("CALC", "concavity",         r"f\s*''\s*\(\s*x\s*\)",
        "medium", "Second derivative / concavity"),
    ("CALC", "tangent_line",      r"m\s*_?\s*(tan|tangent)",
        "high", "Tangent gradient"),
    ("CALC", "optimisation",      r"['\u2032]\s*\(\s*[xPVA]\s*\)\s*=\s*0",
        "high", "Derivative set to zero (max/min)"),
    ("CALC", "optimisation",      r"C\s*'?\s*\(\s*x\s*\)",
        "medium", "Cost function derivative"),

    # --- SEQUENCES ---
    ("SEQ", "arithmetic",         r"T\s*_?\s*n\s*=\s*a\s*\+",
        "high", "Arithmetic term formula"),
    ("SEQ", "arithmetic",         r"S\s*_?\s*n\s*=\s*\\?frac\s*\{\s*n\s*\}\s*\{\s*2\s*\}",
        "high", "Arithmetic sum formula"),
    ("SEQ", "geometric",          r"T\s*_?\s*n\s*=\s*a\s*r\s*\^",
        "high", "Geometric term"),
    ("SEQ", "geometric",          r"r\s*=\s*\\?frac",
        "medium", "Common ratio computed"),
    ("SEQ", "infinite_geometric", r"S\s*_?\s*(\\infty|\u221e)\s*=\s*\\?frac\s*\{\s*a\s*\}\s*\{\s*1\s*-\s*r\s*\}",
        "high", "Sum to infinity formula"),
    ("SEQ", "quadratic_pattern",  r"2\s*a\s*=|second\s+diff",
        "high", "Second difference method"),

    # --- FUNCTIONS ---
    ("FUNC", "hyperbola_asymptote", r"(vertical|horiz)\w*\s+asymptote|y\s*=\s*[-\d]+\s*;.*x\s*=\s*[-\d]+",
        "medium", "Asymptotes stated"),
    ("FUNC", "parabola_tp",       r"x\s*=\s*\\?frac\s*\{\s*-?\s*b\s*\}\s*\{\s*2\s*a\s*\}",
        "high", "Turning point x = -b/2a"),
    ("FUNC", "exponential",       r"y\s*=\s*a\s*[\.·]?\s*2\s*\^|a\s*[\.·]?\s*b\s*\^",
        "medium", "Exponential form"),
    ("FUNC", "inverse",           r"(swap|exchange|interchange).{0,30}x.{0,10}y",
        "high", "Swap x and y for inverse"),
    ("FUNC", "logarithmic",       r"\\?log\s*_?\s*\{?\s*\d",
        "high", "Logarithm notation"),

    # --- ALGEBRA ---
    ("ALG", "quadratic_formula",  r"x\s*=\s*\\?frac\s*\{\s*-?\s*b\s*\\?pm",
        "high", "Quadratic formula substitution"),
    ("ALG", "factorisation",      r"\(\s*x\s*[-+]\s*\d+\s*\)\s*\(\s*x\s*[-+]\s*\d+\s*\)",
        "high", "Factored form"),
    ("ALG", "inequality",         r"(critical\s+values?|CV\s*:)",
        "high", "Inequality critical values"),
    ("ALG", "simultaneous",       r"\(1\)\s*in\s*\(2\)|substitution",
        "medium", "Simultaneous substitution"),
    ("ALG", "nature_of_roots",    r"[Δ\\]?Delta\s*=|discriminant|b\s*\^\s*2\s*-\s*4\s*a\s*c",
        "high", "Discriminant"),

    # --- TRIGONOMETRY ---
    ("TRIG", "compound_angle",    r"cos\s*\(\s*[A-Z]?\s*[-+]\s*[A-Z]?\s*\)\s*=|sin\s*\(\s*[A-Z]?\s*[-+]\s*[A-Z]?\s*\)\s*=",
        "high", "Compound angle expansion"),
    ("TRIG", "double_angle",      r"sin\s*2\s*[A-Za-z]\s*=\s*2\s*sin|cos\s*2\s*[A-Za-z]",
        "high", "Double angle identity"),
    ("TRIG", "reduction",         r"(sin|cos|tan)\s*\(\s*1?[0-9]+\s*[°]?\s*[-+]\s*x",
        "high", "Reduction formula applied"),
    ("TRIG", "general_solution",  r"\+\s*k\s*[\.·]?\s*360|k\s*∈\s*Z|k\s*\\in\s*Z",
        "high", "General solution"),
    ("TRIG", "sine_rule",         r"\\?frac\s*\{\s*a\s*\}\s*\{\s*\\?sin",
        "high", "Sine rule"),
    ("TRIG", "cosine_rule",       r"a\s*\^\s*2\s*=\s*b\s*\^\s*2\s*\+\s*c\s*\^\s*2",
        "high", "Cosine rule"),
    ("TRIG", "area_rule",         r"\\?frac\s*\{\s*1\s*\}\s*\{\s*2\s*\}\s*ab\s*\\?sin",
        "high", "Area rule"),

    # --- ANALYTICAL GEOMETRY ---
    ("AGEO", "gradient",          r"m\s*_?\{?\s*(AB|PQ|RS)\s*\}?\s*=\s*\\?frac",
        "high", "Gradient between two points"),
    ("AGEO", "midpoint",          r"M\s*\(\s*\\?frac\s*\{\s*x\s*_?1",
        "high", "Midpoint formula"),
    ("AGEO", "distance",          r"d\s*=\s*\\?sqrt\s*\{\s*\(.*x\s*_?2",
        "high", "Distance formula"),
    ("AGEO", "circle_equation",   r"\(\s*x\s*-\s*a\s*\)\s*\^\s*2\s*\+\s*\(\s*y\s*-\s*b\s*\)",
        "high", "Circle equation"),
    ("AGEO", "tangent_to_circle", r"m\s*_?\s*(rad|radius|tang)",
        "high", "Tangent to circle gradient"),
    ("AGEO", "angle_inclination", r"tan\s*[θΘ]\s*=|m\s*=\s*tan",
        "high", "Angle of inclination"),

    # --- EUCLIDEAN GEOMETRY ---
    ("EUCL", "with_reasons",      r"with\s+reasons?|met\s+redes",
        "high", "Reason-based proof"),
    ("EUCL", "tangent_chord",     r"tangent[- ]chord|tan[- ]chord|raaklyn[- ]koord",
        "high", "Tangent-chord theorem"),
    ("EUCL", "cyclic_quad",       r"cyclic\s+quad|koordevh|opp\s+∠s\s+of\s+cyc",
        "high", "Cyclic quadrilateral"),
    ("EUCL", "angle_at_centre",   r"(angle|∠)\s+at\s+(the\s+)?cent(re|er)",
        "high", "Angle at centre theorem"),
    ("EUCL", "similarity",        r"\\?triangle\s+[A-Z]+.{0,5}\|{2,3}\s*\\?triangle",
        "high", "Similar triangles"),
    ("EUCL", "proportionality",   r"\\?frac\s*\{\s*(AB|DE|RA)\s*\}\s*\{\s*(BC|EF|AS)\s*\}",
        "medium", "Proportionality from parallel lines"),

    # --- STATISTICS ---
    ("STAT", "mean",              r"\\?bar\s*\{?\s*x\s*\}?\s*=\s*\\?frac",
        "high", "Mean formula"),
    ("STAT", "standard_dev",      r"\\?sigma\s*\^?\s*2?|\u03c3",
        "high", "Standard deviation"),
    ("STAT", "regression",        r"\\?hat\s*\{?\s*y\s*\}?\s*=\s*a\s*\+\s*bx",
        "high", "Regression line y = a + bx"),
    ("STAT", "correlation",       r"r\s*=\s*-?\s*0?[\.\,]\d+",
        "medium", "Correlation coefficient"),
    ("STAT", "ogive_reading",     r"(cumulative\s+frequency|ogive).{0,30}(median|Q1|Q3|IQR)",
        "high", "Ogive quantile reading"),

    # --- PROBABILITY ---
    ("PROB", "venn",              r"P\s*\(\s*A\s*(and|or|\u2229|\u222a)\s*B",
        "high", "Venn diagram probability"),
    ("PROB", "tree",              r"P\s*\(\s*A\s*\)\s*[×x]\s*P\s*\(",
        "medium", "Product rule (tree)"),
    ("PROB", "independence",      r"P\s*\(\s*A\s*\)\s*[×x]\s*P\s*\(\s*B\s*\)\s*=\s*P\s*\(\s*A\s*(and|\u2229)\s*B",
        "high", "Independence test"),
    ("PROB", "counting",          r"\d+\s*!\s*|\d+\s*[×x]\s*\d+\s*[×x]\s*\d+",
        "medium", "Counting/factorial"),

    # --- FINANCE ---
    ("FIN", "compound_interest",  r"P\s*\(\s*1\s*\+\s*i\s*\)\s*\^|A\s*=\s*P\s*\(\s*1",
        "high", "Compound interest formula"),
    ("FIN", "annuity_fv",         r"F\s*=\s*\\?frac\s*\{\s*x\s*\[\s*\(\s*1\s*\+\s*i\s*\)",
        "high", "Future value annuity"),
    ("FIN", "annuity_pv",         r"P\s*=\s*\\?frac\s*\{\s*x\s*\[\s*1\s*-\s*\(\s*1",
        "high", "Present value annuity"),
    ("FIN", "depreciation",       r"A\s*=\s*P\s*\(\s*1\s*-",
        "high", "Depreciation"),
]

sig_df = pd.DataFrame(signatures, columns=[
    "topic_id", "signature_id", "regex", "confidence", "notes"
])
sig_df["taxonomy_version"] = "v2_caps"
sig_df["created_at"] = datetime.now(timezone.utc).isoformat()

out = TAX_DIR / "memo_signatures_v2.csv"
sig_df.to_csv(out, index=False)

print(f"Signatures: {len(sig_df)}")
print(f"Topics covered: {sig_df['topic_id'].nunique()}/10")
print()
print(sig_df.groupby("topic_id").size().to_string())
print()
print(f"Saved: {out}")

Signatures: 57
Topics covered: 10/10

topic_id
AGEO    6
ALG     5
CALC    9
EUCL    6
FIN     4
FUNC    5
PROB    4
SEQ     6
STAT    5
TRIG    7

Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\taxonomy\memo_signatures_v2.csv


In [29]:
sigs = pd.DataFrame([
    {"signature": "first principles", "topic_id": "CALC", "weight": 3},
    {"signature": "dy/dx", "topic_id": "CALC", "weight": 3},
    {"signature": "with reasons", "topic_id": "EUCL", "weight": 2},
    {"signature": "common ratio", "topic_id": "SEQ", "weight": 2},
    {"signature": "asymptote", "topic_id": "FUNC", "weight": 2},
    {"signature": "cyclic quadrilateral", "topic_id": "EUCL", "weight": 3},
    {"signature": "angle in a semicircle", "topic_id": "EUCL", "weight": 3},
    {"signature": "compound interest", "topic_id": "FIN", "weight": 2},
    {"signature": "present value", "topic_id": "FIN", "weight": 2},
    {"signature": "standard deviation", "topic_id": "STAT", "weight": 2},
    {"signature": "least squares", "topic_id": "STAT", "weight": 2},
    {"signature": "reduction formula", "topic_id": "TRIG", "weight": 2},
    {"signature": "identity", "topic_id": "TRIG", "weight": 1},
    {"signature": "equation of the circle", "topic_id": "AGEO", "weight": 3},
    {"signature": "independent events", "topic_id": "PROB", "weight": 2},
])
sigs.to_csv(TAX / "memo_signatures_v2.csv", index=False)
print(sigs)

                 signature topic_id  weight
0         first principles     CALC       3
1                    dy/dx     CALC       3
2             with reasons     EUCL       2
3             common ratio      SEQ       2
4                asymptote     FUNC       2
5     cyclic quadrilateral     EUCL       3
6    angle in a semicircle     EUCL       3
7        compound interest      FIN       2
8            present value      FIN       2
9       standard deviation     STAT       2
10           least squares     STAT       2
11       reduction formula     TRIG       2
12                identity     TRIG       1
13  equation of the circle     AGEO       3
14      independent events     PROB       2


In [30]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

for f in ["subtopics_math_v2.csv", "memo_signatures_v2.csv"]:
    p = TAX_DIR / f
    if p.exists():
        df = pd.read_csv(p)
        print(f"OK  {f:30s}  rows={len(df)}")
    else:
        print(f"MISSING {f}")

OK  subtopics_math_v2.csv           rows=55
OK  memo_signatures_v2.csv          rows=15


In [31]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

sig = pd.read_csv(TAX_DIR / "memo_signatures_v2.csv")
print(f"Rows: {len(sig)}")
print()
print(sig[["topic_id", "signature_id", "confidence"]].to_string(index=False))
print()
print("Topics covered:", sorted(sig["topic_id"].unique().tolist()))

Rows: 15



KeyError: "['signature_id', 'confidence'] not in index"

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

sig = pd.read_csv(TAX_DIR / "memo_signatures_v2.csv")
print("Columns:", list(sig.columns))
print("Rows:", len(sig))
print()
print(sig.head(3).to_string())

Columns: ['signature', 'topic_id', 'weight']
Rows: 15

          signature topic_id  weight
0  first principles     CALC       3
1             dy/dx     CALC       3
2      with reasons     EUCL       2


In [32]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

# Simple substring signatures. No regex metacharacters. No escaping.
# These are matched with `in` (case-insensitive), not re.search.
signatures = [
    # --- CALCULUS ---
    ("CALC", "first_principles",  "lim",                 "high"),
    ("CALC", "dy_dx",             "dy/dx",               "high"),
    ("CALC", "D_x_operator",      "d",                   "low"),  # placeholder, see note below
    ("CALC", "second_derivative", "f''(x)",              "medium"),
    ("CALC", "tangent_gradient",  "tangent",             "high"),
    ("CALC", "derivative_zero",   "= 0",                 "low"),
    ("CALC", "differentiate",     "d/dx",                "high"),

    # --- SEQUENCES ---
    ("SEQ", "arithmetic_term",    "T_n = a +",           "high"),
    ("SEQ", "arithmetic_sum",     "S_n = n/2",           "high"),
    ("SEQ", "geometric_term",     "T_n = a r",           "high"),
    ("SEQ", "geometric_term",     "T_n = ar",            "high"),
    ("SEQ", "common_ratio",       "common ratio",        "high"),
    ("SEQ", "sum_to_infinity",    "sum to infinity",     "high"),
    ("SEQ", "sum_to_infinity",    "S",                   "low"),
    ("SEQ", "second_difference",  "second difference",   "high"),
    ("SEQ", "quadratic_pattern",  "quadratic pattern",   "high"),

    # --- FUNCTIONS ---
    ("FUNC", "asymptotes",        "asymptote",           "high"),
    ("FUNC", "turning_point",     "-b",                  "low"),
    ("FUNC", "inverse_swap",      "swap",                "high"),
    ("FUNC", "inverse_swap",      "interchange",         "high"),
    ("FUNC", "logarithm",         "log",                 "high"),
    ("FUNC", "exponential",       "exponential",         "high"),
    ("FUNC", "parabola",          "parabola",            "high"),
    ("FUNC", "hyperbola",         "hyperbola",           "high"),
    ("FUNC", "domain_range",      "domain",              "medium"),
    ("FUNC", "domain_range",      "range",               "medium"),

    # --- ALGEBRA ---
    ("ALG", "quadratic_formula",  "-b",                  "low"),
    ("ALG", "factorisation",      "(",                   "low"),
    ("ALG", "critical_values",    "critical values",     "high"),
    ("ALG", "critical_values",    "CV",                  "medium"),
    ("ALG", "discriminant",       "discriminant",        "high"),
    ("ALG", "simultaneous",       "substitution",        "medium"),
    ("ALG", "inequality",         "inequality",          "high"),
    ("ALG", "nature_of_roots",    "nature of roots",     "high"),

    # --- TRIGONOMETRY ---
    ("TRIG", "compound_angle",    "cos(",                "low"),
    ("TRIG", "compound_angle",    "compound angle",      "high"),
    ("TRIG", "double_angle",      "double angle",        "high"),
    ("TRIG", "reduction",         "reduction",           "high"),
    ("TRIG", "general_solution",  "general solution",    "high"),
    ("TRIG", "general_solution",  "360",                 "medium"),
    ("TRIG", "sine_rule",         "sine rule",           "high"),
    ("TRIG", "cosine_rule",       "cosine rule",         "high"),
    ("TRIG", "area_rule",         "area rule",           "high"),
    ("TRIG", "identity_proof",    "identity",            "high"),

    # --- ANALYTICAL GEOMETRY ---
    ("AGEO", "gradient",          "gradient",            "high"),
    ("AGEO", "midpoint",          "midpoint",            "high"),
    ("AGEO", "distance",          "distance",            "high"),
    ("AGEO", "circle_equation",   "circle",              "high"),
    ("AGEO", "tangent_to_circle", "tangent",             "medium"),
    ("AGEO", "angle_inclination", "inclination",         "high"),
    ("AGEO", "parallel_perp",     "parallel",            "medium"),
    ("AGEO", "parallel_perp",     "perpendicular",       "medium"),

    # --- EUCLIDEAN GEOMETRY ---
    ("EUCL", "with_reasons",      "with reasons",        "high"),
    ("EUCL", "tangent_chord",     "tangent",             "medium"),
    ("EUCL", "tangent_chord",     "chord",               "high"),
    ("EUCL", "cyclic_quad",       "cyclic",              "high"),
    ("EUCL", "angle_at_centre",   "centre",              "medium"),
    ("EUCL", "similarity",        "similar",             "high"),
    ("EUCL", "similarity",        "|||",                 "high"),
    ("EUCL", "proportionality",   "proportional",        "medium"),
    ("EUCL", "midpoint_theorem",  "midpoint",            "medium"),
    ("EUCL", "pythagoras",        "pythagoras",          "high"),

    # --- STATISTICS ---
    ("STAT", "mean",              "mean",                "medium"),
    ("STAT", "standard_dev",      "standard deviation",  "high"),
    ("STAT", "regression",        "regression",          "high"),
    ("STAT", "correlation",       "correlation",         "high"),
    ("STAT", "ogive_reading",     "ogive",               "high"),
    ("STAT", "ogive_reading",     "cumulative",          "high"),
    ("STAT", "box_whisker",       "box and whisker",     "high"),
    ("STAT", "iqr",               "IQR",                 "high"),
    ("STAT", "iqr",               "interquartile",       "high"),

    # --- PROBABILITY ---
    ("PROB", "venn",              "venn",                "high"),
    ("PROB", "tree",              "tree diagram",        "high"),
    ("PROB", "independence",      "independent",         "high"),
    ("PROB", "mutually_exclusive","mutually exclusive",  "high"),
    ("PROB", "counting",          "factorial",           "medium"),
    ("PROB", "counting",          "combination",         "medium"),
    ("PROB", "complement",        "complementary",       "medium"),

    # --- FINANCE ---
    ("FIN", "compound_interest",  "compound",            "high"),
    ("FIN", "simple_interest",    "simple interest",     "high"),
    ("FIN", "depreciation",       "depreciat",           "high"),
    ("FIN", "annuity_fv",         "future value",        "high"),
    ("FIN", "annuity_pv",         "present value",       "high"),
    ("FIN", "nominal_effective",  "nominal",             "high"),
    ("FIN", "nominal_effective",  "effective",           "high"),
    ("FIN", "time_line",          "time line",           "high"),
]

sig = pd.DataFrame(signatures, columns=["topic_id", "signature_id", "pattern", "confidence"])
sig["taxonomy_version"] = "v2_caps"
sig["match_method"] = "substring_case_insensitive"

out = TAX_DIR / "memo_signatures_v2.csv"
sig.to_csv(out, index=False)

print(f"Wrote {len(sig)} signatures")
print()
print(sig.groupby("topic_id").size().to_string())
print()
print("Topics covered:", sorted(sig["topic_id"].unique().tolist()))

expected = {"ALG","SEQ","FIN","FUNC","CALC","PROB","STAT","AGEO","TRIG","EUCL"}
missing = expected - set(sig["topic_id"].unique())
print("Missing topics:", sorted(missing) if missing else "none")
print()
print(f"Saved: {out}")

Wrote 86 signatures

topic_id
AGEO     8
ALG      8
CALC     7
EUCL    10
FIN      8
FUNC    10
PROB     7
SEQ      9
STAT     9
TRIG    10

Topics covered: ['AGEO', 'ALG', 'CALC', 'EUCL', 'FIN', 'FUNC', 'PROB', 'SEQ', 'STAT', 'TRIG']
Missing topics: none

Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\taxonomy\memo_signatures_v2.csv


In [33]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

sig = pd.read_csv(TAX_DIR / "memo_signatures_v2.csv")
print(f"Total: {len(sig)}")
print(f"Columns: {list(sig.columns)}")
print()
print(sig.groupby(["topic_id", "confidence"]).size().to_string())

Total: 86
Columns: ['topic_id', 'signature_id', 'pattern', 'confidence', 'taxonomy_version', 'match_method']

topic_id  confidence
AGEO      high          5
          medium        3
ALG       high          4
          low           2
          medium        2
CALC      high          4
          low           2
          medium        1
EUCL      high          6
          medium        4
FIN       high          8
FUNC      high          7
          low           1
          medium        2
PROB      high          4
          medium        3
SEQ       high          8
          low           1
STAT      high          8
          medium        1
TRIG      high          8
          low           1
          medium        1


In [34]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TAX_DIR = ROOT / "data" / "processed" / "taxonomy"

sig = pd.read_csv(TAX_DIR / "memo_signatures_v2.csv")

# Usage rules:
#   PRIMARY    — fires on its own, assigns topic
#   SECONDARY  — fires only if no PRIMARY matches
#   FALLBACK   — fires only if topic already assigned by context (parent question)
usage_map = {
    "low": "FALLBACK",      # never fires on its own
    "medium": "SECONDARY",  # fires if no high match
    "high": "PRIMARY",      # fires freely
}

sig["usage_rule"] = sig["confidence"].map(usage_map)
sig.to_csv(TAX_DIR / "memo_signatures_v2.csv", index=False)

print(sig.groupby(["usage_rule", "topic_id"]).size().unstack(fill_value=0))

topic_id    AGEO  ALG  CALC  EUCL  FIN  FUNC  PROB  SEQ  STAT  TRIG
usage_rule                                                         
FALLBACK       0    2     2     0    0     1     0    1     0     1
PRIMARY        5    4     4     6    8     7     4    8     8     8
SECONDARY      3    2     1     4    0     2     3    0     1     1


In [35]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

# Where do 2023-2025 mapped outputs currently live?
for year in [2023, 2024, 2025]:
    # Check the mapped directory first
    candidates = [
        ROOT / "data" / "processed" / "mapped" / str(year) / f"question_topic_map_{year}.csv",
        ROOT / "data" / "processed" / "mapped" / f"question_topic_map_{year}.csv",
        ROOT / "data" / "processed" / "mapped" / f"question_topic_map_exams_v1.csv",
    ]
    found = None
    for c in candidates:
        if c.exists():
            found = c
            break
    if found:
        df = pd.read_csv(found)
        print(f"{year}: {found.name}  rows={len(df)}  cols={list(df.columns)[:6]}...")
    else:
        print(f"{year}: NOT FOUND — check where original mapped files live")

2023: question_topic_map_exams_v1.csv  rows=261  cols=['document_id', 'year', 'paper', 'document_type', 'source_text_file', 'extraction_quality']...
2024: question_topic_map_exams_v1.csv  rows=261  cols=['document_id', 'year', 'paper', 'document_type', 'source_text_file', 'extraction_quality']...
2025: question_topic_map_exams_v1.csv  rows=261  cols=['document_id', 'year', 'paper', 'document_type', 'source_text_file', 'extraction_quality']...


In [36]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
COMBINED = ROOT / "data" / "processed" / "mapped" / "question_topic_map_exams_v1.csv"

df = pd.read_csv(COMBINED)
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
print("Rows per year:")
print(df.groupby("year").size().to_string())
print()
print("Rows per year and paper:")
print(df.groupby(["year", "paper"]).size().to_string())
print()
print("First 3 rows:")
print(df.head(3).to_string())

Rows: 261
Columns: ['document_id', 'year', 'paper', 'document_type', 'source_text_file', 'extraction_quality', 'question_number', 'subquestion', 'marks', 'marks_all_found', 'question_text', 'question_char_count', 'segment_index', 'segmentation_status', 'question_id', 'topic', 'subtopic', 'mapping_method', 'rule_id', 'mapping_confidence', 'taxonomy_version']

Rows per year:
year
2021     1
2023    95
2024    78
2025    87

Rows per year and paper:
year  paper
2023  P1       49
      P2       46
2024  P1       37
      P2       41
2025  P1       44
      P2       43

First 3 rows:
              document_id  year paper document_type                                        source_text_file extraction_quality  question_number subquestion  marks marks_all_found                                               question_text  question_char_count  segment_index segmentation_status                        question_id                topic             subtopic   mapping_method            rule_id mappin

In [37]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
COMBINED = ROOT / "data" / "processed" / "mapped" / "question_topic_map_exams_v1.csv"

df = pd.read_csv(COMBINED)
stray = df[df["year"] == 2021]

print(f"Stray 2021 rows: {len(stray)}")
print()
if len(stray) > 0:
    cols_to_show = ["document_id", "paper", "question_number", "subquestion", "question_text"]
    cols_to_show = [c for c in cols_to_show if c in stray.columns]
    print(stray[cols_to_show].to_string(index=False))
    print()
    print("Full row:")
    print(stray.iloc[0].to_dict())

Stray 2021 rows: 1

                            document_id paper  question_number subquestion                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [38]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
COMBINED = ROOT / "data" / "processed" / "mapped" / "question_topic_map_exams_v1.csv"

df = pd.read_csv(COMBINED)
before = len(df)

# Drop rows that are not real exam questions.
# A real row has: paper set, question_number set, document_type == 'exam'.
drop_mask = (
    df["paper"].isna()
    | df["question_number"].isna()
    | (df.get("document_type", "") != "exam")
)
dropped = df[drop_mask]

# Backup dropped rows for audit
if len(dropped) > 0:
    backup = ROOT / "data" / "processed" / "mapped" / "dropped_rows_from_combined_v1.csv"
    dropped.to_csv(backup, index=False)
    print(f"Backup of dropped rows: {backup.relative_to(ROOT)}")
    print(f"Dropped {len(dropped)} rows:")
    for _, r in dropped.iterrows():
        print(f"  - {r['document_id']}  type={r.get('document_type')}  qnum={r.get('question_number')}")
    print()

df_clean = df[~drop_mask].copy()
df_clean.to_csv(COMBINED, index=False)
print(f"Combined file: {before} -> {len(df_clean)} rows")

Backup of dropped rows: data\processed\mapped\dropped_rows_from_combined_v1.csv
Dropped 1 rows:
  - grade12_mathematics_exam_guideline_2021  type=guideline  qnum=nan

Combined file: 261 -> 260 rows


In [39]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
COMBINED = ROOT / "data" / "processed" / "mapped" / "question_topic_map_exams_v1.csv"
OUT_DIR = ROOT / "data" / "processed" / "mapped"

df = pd.read_csv(COMBINED)
df = df[df["year"].isin([2023, 2024, 2025])].copy()

print(f"Rows to split: {len(df)}")
print()

for year in [2023, 2024, 2025]:
    sub = df[df["year"] == year].copy()
    year_dir = OUT_DIR / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)
    out = year_dir / f"question_topic_map_{year}.csv"
    sub.to_csv(out, index=False)
    p1 = len(sub[sub["paper"] == "P1"])
    p2 = len(sub[sub["paper"] == "P2"])
    print(f"{year}: {len(sub)} rows (P1={p1}, P2={p2})")

Rows to split: 260

2023: 95 rows (P1=49, P2=46)
2024: 78 rows (P1=37, P2=41)
2025: 87 rows (P1=44, P2=43)


In [40]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

print("Per-year mapped files (v1):")
print()
for year in [2021, 2022, 2023, 2024, 2025]:
    p = ROOT / "data" / "processed" / "mapped" / str(year) / f"question_topic_map_{year}.csv"
    if p.exists():
        df = pd.read_csv(p)
        p1 = (df["paper"] == "P1").sum()
        p2 = (df["paper"] == "P2").sum()
        has_nan = df["question_number"].isna().sum()
        flag = "  ⚠ contains NaN question_number" if has_nan else ""
        print(f"  {year}: {len(df):>4} rows  (P1={p1:>3}, P2={p2:>3}){flag}")
    else:
        print(f"  {year}: MISSING")

# Also confirm combined is clean
combined = pd.read_csv(ROOT / "data" / "processed" / "mapped" / "question_topic_map_exams_v1.csv")
print()
print(f"Combined file rows: {len(combined)}")
print(f"NaN paper rows:     {combined['paper'].isna().sum()}")
print(f"NaN qnum rows:      {combined['question_number'].isna().sum()}")

Per-year mapped files (v1):

  2021:  119 rows  (P1= 57, P2= 62)
  2022:  116 rows  (P1= 56, P2= 60)
  2023:   95 rows  (P1= 49, P2= 46)
  2024:   78 rows  (P1= 37, P2= 41)
  2025:   87 rows  (P1= 44, P2= 43)

Combined file rows: 260
NaN paper rows:     0
NaN qnum rows:      0


In [41]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
NOTES = ROOT / "docs" / "pilot" / "mapped_v1_known_issues.md"

existing = NOTES.read_text(encoding="utf-8") if NOTES.exists() else ""

addendum = """

## Stray 2021 guideline row — RESOLVED
- Row `grade12_mathematics_exam_guideline_2021` was a document-level leak
  of the 2021 exam guideline PDF into the exam mapping table.
- Dropped from combined file (261 -> 260 rows).
- Backup at: data/processed/mapped/dropped_rows_from_combined_v1.csv
- Root cause: guideline documents were segmented as if they were papers.
  This should not recur if N03 excludes non-exam document types.
- Action for notebook 14: verify loader rejects any row where
  document_type != 'exam' OR paper is NaN OR question_number is NaN.
"""

NOTES.write_text(existing + addendum, encoding="utf-8")
print(f"Updated {NOTES.relative_to(ROOT)}")

Updated docs\pilot\mapped_v1_known_issues.md
